In [ ]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Instalar las librerías necesarias (el signo ! es obligatorio en Colab)
!pip install sdv polars pyarrow pandas

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.4/215.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.7/213.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 4.7 MB/s eta 0:00:00


In [ ]:
import polars as pl
import pandas as pd

ruta_entrada = '/content/drive/MyDrive/programacion recurrente /postpartum_depression_processed.parquet'

# Leer con Polars y convertir a Pandas
df_polars = pl.read_parquet(ruta_entrada)
df_pandas = df_polars.to_pandas()

print(f"Dataset cargado con éxito. Filas actuales: {len(df_pandas)}")

Dataset cargado con éxito. Filas actuales: 1503


In [ ]:
from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

# Autodetectar la estructura de los datos
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_pandas)

print("Entrenando el modelo generativo (esto tomará unos segundos)...")
# Entrenar el modelo con los registros originales
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(df_pandas)
print("¡Modelo entrenado!")

Entrenando el modelo generativo (esto tomará unos segundos)...


/usr/local/lib/python3.13/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.13/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


¡Modelo entrenado!


In [ ]:
# Calcular cuántas filas faltan para llegar a 1,000,000
registros_actuales = len(df_pandas)
registros_a_generar = 1000000 - registros_actuales
print(f"Generando {registros_a_generar} filas nuevas. Esto va a demorar unos minutos...")

# Generar los datos sintéticos
synthetic_data = synthesizer.sample(num_rows=registros_a_generar)

# Unir la data original con la nueva
df_final_pandas = pd.concat([df_pandas, synthetic_data], ignore_index=True)

print(f"¡Data augmentation completado! Filas totales: {len(df_final_pandas)}")

Generando 998497 filas nuevas. Esto va a demorar unos minutos...
¡Data augmentation completado! Filas totales: 1000000


In [ ]:
import os
import polars as pl

# 1. Extraer automáticamente la carpeta exacta desde tu ruta que sí funcionó
carpeta_correcta = os.path.dirname(ruta_entrada)

# 2. Crear la nueva ruta uniendo esa carpeta con el nombre del nuevo archivo
ruta_salida_dinamica = os.path.join(carpeta_correcta, 'postpartum_depression_1M.parquet')

# 3. Convertir a Polars y guardar
df_final_polars = pl.from_pandas(df_final_pandas)
df_final_polars.write_parquet(ruta_salida_dinamica, compression='snappy')

print(f"¡Listo! Dataset de 1 millón de registros guardado en:\n{ruta_salida_dinamica}")

¡Listo! Dataset de 1 millón de registros guardado en:
/content/drive/MyDrive/programacion recurrente /postpartum_depression_1M.parquet
